# 2.6 Mould check — with / without cobweb
Reproducible paired YOLO11n experiment. Original labels survive the audit; fixed **train / validation / test** groups are shared by both runs. Without-cobweb removes whole training photos (and duplicate groups) containing any cobweb label. Validation selects the best epoch; test is scored only after training. **syldd is excluded.** Myceliummonitor is a separate future experiment, never part of this comparison.

First run with `RUN_TRAINING=False`. Add `d005-clean-dishes` and `mould-test-photos` as Kaggle inputs, enable Internet and the `ROBOFLOW_API_KEY` secret. Run the audit, inspect saved label grids, duplicate groups, dish IDs and split counts. Fill `GROUP_OVERRIDES` for related Baibua dishes/sequences that filenames do not identify; then acknowledge `GROUPING_REVIEWED=True`. A hash cannot prove that different views show different dishes.

Record the printed Roboflow versions in `RF_VERSIONS` and rerun with those fixed numbers before training. The newest version is discovery only. Set a fresh `COMPARISON_ID`, Kaggle version and `RUN_TRAINING=True` only after review. GPU: T4 or P100. No full training is launched by default.

Baibua data: credit baibua / Roboflow (CC BY 4.0; verify downloaded licence metadata). Clean-photo backgrounds differ from contaminated-photo backgrounds; scores may reflect source cues. Small cobweb counts limit conclusions. Output is “possible mould, check it”, never a diagnosis.


In [ ]:
%pip install -q ultralytics==8.3.221 roboflow==1.2.11 PyYAML==6.0.2


In [ ]:
import json, random, re, shutil, hashlib, math, sys, subprocess, platform
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

SETS = {"wwzez": ("baibua", "mold-detection-training-wwzez"),
        "f3os8": ("baibua", "mold-detect-s-f3os8"), "4n7zr": ("baibua", "molds-4n7zr")}
RF_VERSIONS = {s: None for s in SETS}  # discovery only: replace None with printed version numbers
CLASS_MAP = {"trichoderma": 0, "mold": 1, "mould": 1, "cobweb": 1, "neurospora": 1}
OUR_NAMES = {0: "Trichoderma", 1: "mould"}
CLEAN_DIR = Path("/kaggle/input/d005-clean-dishes")
BENCH_DIR = Path("/kaggle/input/mould-test-photos")
CLEAN_ID_REGEX = r"(\d{6}_[A-Za-z]+_Tv)"
# Keys: '<source>/<original filename>'; values: shared dish/sequence ID across sources.
GROUP_OVERRIDES = {}
GROUPING_REVIEWED = False
RUN_TRAINING = False
COMPARISON_ID = "mould_r01_cobweb"  # never reuse after starting training
KAGGLE_VERSION = "FILL_ME"
SEED, HASH_DISTANCE = 0, 4
WORK = Path("/kaggle/working")
OUT = WORK / "cv_runs" / "mould" / COMPARISON_ID
AUDIT = OUT / "audit"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TRAIN_SETTINGS = dict(epochs=50, patience=15, imgsz=640, batch=16, seed=SEED,
    deterministic=True, workers=0, device=0, optimizer="AdamW", lr0=0.001,
    weight_decay=0.0005, amp=False, plots=True, save=True)
MODEL = "yolo11n.pt"
AUDIT.mkdir(parents=True, exist_ok=True)
VALIDATED = False

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False))
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def safe(path):
    if "syldd" in str(path).lower():
        raise ValueError(f"syldd is excluded from training/tuning: {path}")
def images(root):
    safe(root)
    ps = sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT)
    if not ps: raise ValueError(f"No images under {root}; check Kaggle input path")
    for p in ps: safe(p)
    return ps


## 1. Download and audit original annotations
Missing, malformed, unknown, empty contaminant labels and unreadable images stop preparation. Only explicitly clean d005 photos receive empty labels. No polygon-to-box conversion or silent label dropping. Each source version and image/label SHA256 is saved. Repeated downloads reuse the recorded version; changing a version requires a new comparison ID.


In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
version_file = AUDIT / "source_versions.json"
previous = json.loads(version_file.read_text()) if version_file.exists() else {}
downloads, resolved = {}, {}
for short, (ws, project) in SETS.items():
    proj = rf.workspace(ws).project(project)
    v = RF_VERSIONS[short] or previous.get(short) or max(int(str(x.version).split("/")[-1]) for x in proj.versions())
    resolved[short] = int(v)
    loc = WORK / "raw" / short / str(v)
    if not (loc / "data.yaml").exists():
        proj.version(v).download("yolov11", location=str(loc), overwrite=False)
    cfg = yaml.safe_load((loc / "data.yaml").read_text())
    names = cfg["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k,v in names.items()}
    downloads[short] = (loc, names)
    print(short, "version", v, "original labels", names)
save_json(version_file, resolved)


In [ ]:
def read_boxes(label, names):
    if not label.is_file(): raise ValueError(f"Missing label: {label}")
    boxes = []
    for lineno, line in enumerate(label.read_text().splitlines(), 1):
        if not line.strip(): continue
        a = line.split()
        if len(a) != 5: raise ValueError(f"Expected YOLO box, {label}:{lineno}")
        cls, x, y, w, h = map(float, a)
        if not all(math.isfinite(t) for t in (cls,x,y,w,h)) or cls != int(cls) or int(cls) not in names:
            raise ValueError(f"Invalid class/numbers: {label}:{lineno}")
        name = str(names[int(cls)]).strip()
        if name.lower() not in CLASS_MAP: raise ValueError(f"Unmapped original class {name}: {label}")
        if not (0 < w <= 1 and 0 < h <= 1 and x-w/2 >= -1e-5 and y-h/2 >= -1e-5 and x+w/2 <= 1+1e-5 and y+h/2 <= 1+1e-5):
            raise ValueError(f"Out-of-image box: {label}:{lineno}")
        boxes.append((name,x,y,w,h))
    if not boxes: raise ValueError(f"Empty contaminant label needs review: {label}")
    return boxes

def record(path, src, boxes, dish=None, label=None, benchmark=False, positive=None):
    safe(path)
    return dict(id=hashlib.sha256(f"{src}/{path.name}".encode()).hexdigest()[:20],
        path=str(path), original_filename=path.name, source=src, dish=dish,
        boxes=boxes, has_cobweb=any(b[0].lower()=="cobweb" for b in boxes),
        clean=src=="d005", benchmark=benchmark, positive=positive,
        image_sha256=sha(path), label_path=str(label) if label else None, label_sha256=sha(label) if label else None)

def benchmark_info(path, root):
    # The first directory under BENCH_DIR is the user-supplied image label.
    parts = Path(path).relative_to(root).parts
    if len(parts) < 2 or parts[0] not in {'clean', 'cobweb', 'mould'}:
        raise ValueError(f"Expected clean/, cobweb/ or mould/ benchmark folder: {path}")
    category = parts[0]
    return dict(benchmark_category=category, positive=category != 'clean',
                has_cobweb=category == 'cobweb', clean=category == 'clean',
                label_basis='folder_only')

VALIDATED = False
raw = []
for src, (root, names) in downloads.items():
    for p in images(root):
        if p.parent.name != "images": continue
        lab = p.parent.parent / "labels" / (p.stem + ".txt")
        dish = GROUP_OVERRIDES.get(f"{src}/{p.name}")
        raw.append(record(p, src, read_boxes(lab,names), dish, lab))
if not raw: raise ValueError("No box-labelled source images")
if any(not any(r["source"]==s for r in raw) for s in SETS):
    raise ValueError("A source has no box-labelled images; inspect its download")
unknown_overrides = set(GROUP_OVERRIDES)-{f"{r['source']}/{r['original_filename']}" for r in raw}
if unknown_overrides: raise ValueError(f"Unknown grouping override keys: {unknown_overrides}")
clean_files = images(CLEAN_DIR)
by_folder = defaultdict(set)
for p in clean_files:
    m = re.search(CLEAN_ID_REGEX, str(p.relative_to(CLEAN_DIR)))
    if m: by_folder[p.parent].add(m.group(1))
for p in clean_files:
    m = re.search(CLEAN_ID_REGEX, str(p.relative_to(CLEAN_DIR)))
    ids = {m.group(1)} if m else by_folder[p.parent]
    if len(ids)!=1: raise ValueError(f"Ambiguous clean dish: {p}")
    raw.append(record(p, "d005", [], "d005/"+next(iter(ids))))
bench = []
for p in images(BENCH_DIR):
    info = benchmark_info(p, BENCH_DIR)
    r = record(p, "benchmark", [], benchmark=True, positive=info['positive'])
    r.update(info)
    bench.append(r)
benchmark_counts = dict(Counter(r['benchmark_category'] for r in bench))
save_json(AUDIT/'benchmark_folder_counts.json', benchmark_counts)
print('Benchmark folder labels (image-level only):', benchmark_counts)
if len({r['id'] for r in raw+bench}) != len(raw+bench):
    raise ValueError("Repeated source filenames: provide unique provenance before continuing")
original_counts = []
for src in SETS:
    rs = [r for r in raw if r['source']==src]
    for cls in sorted({b[0] for r in rs for b in r['boxes']}):
        original_counts.append(dict(source=src, original_class=cls,
            images=sum(any(b[0]==cls for b in r['boxes']) for r in rs),
            boxes=sum(b[0]==cls for r in rs for b in r['boxes'])))
pd.DataFrame(original_counts).to_csv(AUDIT/'original_class_counts.csv', index=False)
save_json(AUDIT/'original_records.json', raw+bench)
display(pd.DataFrame(original_counts))
print("Raw photos",dict(Counter(r['source'] for r in raw)), "benchmark",len(bench))


## 2. Duplicate groups, dish groups, benchmark overlap
64-bit perceptual hash (32×32 DCT, top 8×8, median threshold), Hamming distance ≤4; also check rotations/reflections, exact bytes, and Roboflow original-name identities. Connected components preserve transitive links across sources, clean dishes and benchmark photos. One representative per duplicate group is kept; every original filename remains in the audit. Cobweb status is the union of labels in that duplicate group. Conflicting clean/contaminated groups or different original-class sets stop the run for annotation review. Review near-duplicate montages: hashes can join similar dishes or miss different views. Benchmark-linked **whole dish groups go to test**, even if that makes test larger than 20%.


In [ ]:
def phashes(path):
    with Image.open(path) as im:
        if im.getexif().get(274,1) not in (None, 0, 1): raise ValueError(f"EXIF orientation needs review: {path}")
        im = im.convert('L').resize((32,32), Image.Resampling.LANCZOS)
        a = np.asarray(im, dtype=float)
    dct = 2*np.cos(np.pi*np.arange(8)[:,None]*(2*np.arange(32)[None,:]+1)/64)
    result=[]
    for base in (a, np.fliplr(a)):
        for k in range(4):
            d = dct @ np.rot90(base,k) @ dct.T
            bits = (d > np.median(d)).ravel()
            result.append(sum(int(v)<<i for i,v in enumerate(bits)))
    return result

class UnionFind:
    def __init__(self,n): self.p=list(range(n))
    def find(self,i):
        while self.p[i]!=i:
            self.p[i]=self.p[self.p[i]]; i=self.p[i]
        return i
    def join(self,i,j): self.p[self.find(i)]=self.find(j)

def group_records(records):
    uf=UnionFind(len(records)); known={}; hashes=[]
    for i,r in enumerate(records):
        hs=phashes(r['path']); hashes.append(hs)
        # Different source exports of the same basename also stay together.
        origin=Path(r['original_filename']).stem.split('.rf.')[0]
        for key in [('sha',r['image_sha256']),('origin',origin)]:
            if key in known: uf.join(i,known[key])
            known[key]=i
        for j in range(i):
            if min((h^g).bit_count() for h in hs for g in hashes[j]) <= HASH_DISTANCE:
                uf.join(i,j)
    groups=defaultdict(list)
    for i,r in enumerate(records): groups[uf.find(i)].append(r)
    for members in groups.values():
        gid=min(r['id'] for r in members)
        for r in members: r['duplicate_group']=gid
    # Union duplicate links and biological dish/sequence links, across all sources.
    dishes={}
    for i,r in enumerate(records):
        if r['dish']:
            if r['dish'] in dishes: uf.join(i,dishes[r['dish']])
            dishes[r['dish']]=i
    connected=defaultdict(list)
    for i,r in enumerate(records): connected[uf.find(i)].append(r)
    for members in connected.values():
        gid=min(r['id'] for r in members)
        overlap=any(r['benchmark'] for r in members)
        for r in members: r.update(group=gid, benchmark_overlap=overlap)
    return groups

duplicates=group_records(raw+bench)
rows=[]; unique=[]; conflicts=[]
for members in duplicates.values():
    candidates=[r for r in members if not r['benchmark']]
    labels={tuple(sorted({b[0].lower() for b in r['boxes']})) for r in candidates}
    if len(labels)>1: conflicts.append([r['original_filename'] for r in candidates])
    kept=min(candidates,key=lambda r:(-len(r['boxes']),r['id'])) if candidates else None
    if kept:
        kept['has_cobweb']=any(r['has_cobweb'] for r in candidates)
        unique.append(kept)
    for r in members:
        rows.append({k:v for k,v in r.items() if k!='boxes'} | {'kept':r is kept, 'duplicate_size':len(members)})
pd.DataFrame(rows).to_csv(AUDIT/'duplicate_and_overlap_manifest.csv',index=False)
save_json(AUDIT/'label_conflicts.json',conflicts)
print("Duplicate groups:",sum(len(g)>1 for g in duplicates.values()),"representatives:",len(unique))
print("Cobweb photos before/after dedup:",sum(r['has_cobweb'] for r in raw),sum(r['has_cobweb'] for r in unique))
if conflicts: raise ValueError("Conflicting duplicate annotations: inspect label_conflicts.json before training")


## 3. Fixed group splits and validation gates
A deterministic group allocation targets 60/20/20 within clean, cobweb and other contaminated strata. Benchmark groups are forced to test first. Empty or unsuitable splits stop rather than relaxing grouping. The saved manifest is immutable: changed input/settings require a new comparison ID. Test clean photos are included in all/non-cobweb detection evaluation with explicit empty ground-truth labels; the cobweb subset consists of box-labelled cobweb photos only.


In [ ]:
def assign_splits(records, seed=SEED):
    groups=defaultdict(list)
    for r in records: groups[r['group']].append(r)
    assignment={g:'test' for g,rs in groups.items() if any(r['benchmark_overlap'] for r in rs)}
    strata=defaultdict(list)
    for g,rs in groups.items():
        kind='clean' if all(r['clean'] for r in rs) else 'cobweb' if any(r['has_cobweb'] for r in rs) else 'other'
        strata[kind].append(g)
    for kind,gs in sorted(strata.items()):
        gs=sorted(gs); random.Random(seed).shuffle(gs)
        available=[g for g in gs if g not in assignment]
        n=max(1,round(.2*len(gs)))
        forced=sum(g in assignment for g in gs)
        nt=max(0,n-forced)
        for i,g in enumerate(available): assignment[g]='test' if i<nt else 'val' if i<nt+n else 'train'
    return {r['id']:assignment[r['group']] for r in records}

def validate_splits(records):
    for field in ('group','duplicate_group','dish'):
        seen=defaultdict(set)
        for r in records:
            if r[field]: seen[r[field]].add(r['split'])
        if any(len(s)>1 for s in seen.values()): raise ValueError(f"Leakage through {field}")
    for side in ('train','val','test'):
        rs=[r for r in records if r['split']==side]
        if not any(r['clean'] for r in rs) or not any(r['boxes'] for r in rs):
            raise ValueError(f"{side} needs clean and box-labelled positive photos; review groups")
        if {CLASS_MAP[b[0].lower()] for r in rs for b in r['boxes']} != {0,1}:
            raise ValueError(f"{side} lacks a mapped class; review labels/groups")
    for r in records:
        safe(r['path'])
        if r['benchmark_overlap'] and r['split']!='test': raise ValueError("Benchmark overlap in training/tuning")
    train=[r for r in records if r['split']=='train']
    if not any(r['has_cobweb'] for r in train): raise ValueError("No training cobweb: comparison would be identical")
    without=[r for r in train if not r['has_cobweb']]
    if not any(r['clean'] for r in without) or {CLASS_MAP[b[0].lower()] for r in without for b in r['boxes']}!={0,1}:
        raise ValueError("Without-cobweb training lacks clean examples or a class")

VALIDATED=False
assignment=assign_splits(unique)
for r in unique: r['split']=assignment[r['id']]
manifest=[{k:v for k,v in r.items() if k!='boxes'} | {'original_classes':sorted({b[0] for b in r['boxes']}),
    'n_boxes':len(r['boxes'])} for r in sorted(unique,key=lambda r:r['id'])]
benchmark_manifest=[{k:v for k,v in r.items() if k!='boxes'} for r in sorted(bench,key=lambda r:r['id'])]
pd.DataFrame(benchmark_manifest).to_csv(OUT/'benchmark_manifest.csv',index=False)
contract=dict(manifest=manifest, benchmark_manifest=benchmark_manifest, source_versions=resolved, seed=SEED, hash_distance=HASH_DISTANCE,
    group_overrides=GROUP_OVERRIDES, class_map=CLASS_MAP)
contract_file=OUT/'split_contract.json'
if contract_file.exists() and json.loads(contract_file.read_text())!=contract:
    raise ValueError("Saved split contract differs: restore inputs/settings or choose a new COMPARISON_ID")
save_json(contract_file,contract)
pd.DataFrame(manifest).to_csv(OUT/'split_manifest.csv',index=False)
validate_splits(unique)
VALIDATED=True
VALIDATED_DIGEST=hashlib.sha256(json.dumps(raw+bench,sort_keys=True).encode()).hexdigest()
summary=pd.DataFrame(manifest).groupby(['split','source']).agg(photos=('id','size'),cobweb_photos=('has_cobweb','sum'),boxes=('n_boxes','sum'))
summary.to_csv(AUDIT/'split_counts.csv'); display(summary)
if not any(r['split']=='test' and r['has_cobweb'] for r in unique):
    print('WARNING: no cobweb test photos; cobweb metrics will be null, comparison is limited.')
print('Automatic validation passed. Review label grids, duplicates and dish grouping before enabling training.')


In [ ]:
# Save original-label samples and multi-member duplicate montages for human review.
def grid(records, target, title):
    fig, axes=plt.subplots(math.ceil(len(records)/4),4,figsize=(16,4*math.ceil(len(records)/4)),squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax,r in zip(axes.flat,records):
        with Image.open(r['path']) as im: im=im.convert('RGB')
        d=ImageDraw.Draw(im); w,h=im.size
        for cls,x,y,bw,bh in r['boxes']:
            d.rectangle(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h),outline='red',width=3)
            d.text(((x-bw/2)*w,(y-bh/2)*h),cls,fill='red')
        ax.imshow(im); ax.set_title(r['original_filename'][:45],fontsize=8)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(target,dpi=110); plt.close(fig)
for src in SETS:
    rs=[r for r in raw if r['source']==src]
    if rs: grid(random.Random(SEED).sample(rs,min(12,len(rs))),AUDIT/f'labels_{src}.png',src)
for gid,members in enumerate(duplicates.values()):
    if len(members)>1:
        for page,start in enumerate(range(0,len(members),12)):
            grid(members[start:start+12],AUDIT/f'duplicate_{gid}_{page}.png','Review duplicate group')
shutil.make_archive(str(OUT)+'_audit','zip',OUT)
print('Audit bundle:',str(OUT)+'_audit.zip')


## 4. Paired training and independent testing (disabled by default)
Both runs start from the same saved pretrained weights and identical explicit settings. Epoch count may differ because early stopping uses the same validation data. Detection P/R use Ultralytics' confidence-curve operating point; mAP50 integrates that curve (`conf=0.001`, NMS IoU=0.7). Image sensitivity means **any predicted contaminant box on a positive photo**, without claiming correct localisation; false-alarm rate is any box on a clean photo. Both use confidence 0.25 / 0.50. Undefined rates and empty detection subsets are null with sample counts.

Benchmark folders are `clean/` (negative), `cobweb/` (positive, cobweb) and `mould/` (positive, non-cobweb). Report all/cobweb/non-cobweb image-level subsets at both thresholds, within each overlap stratum. Non-cobweb includes the clean photos; cobweb-only false-alarm rate is null because that subset has no clean photos. These user-supplied folder labels support image-level results **only**, never detection mAP. Save separate unmatched, overlap-with-heldout, and overlap-with-training/tuning strata; the last should be empty. “Unmatched” means no overlap detected by this audit, not proven independence from pretraining. Recompute MycoGuard on the same filenames before comparing a changed subset. No test-based tuning.

References: [Ultralytics training](https://docs.ultralytics.com/modes/train/), [validation](https://docs.ultralytics.com/modes/val/).


In [ ]:
def build_dataset(folder, parts):
    if folder.exists(): raise ValueError(f"Refusing to overwrite dataset: {folder}")
    for side,rs in parts.items():
        (folder/'images'/side).mkdir(parents=True)
        (folder/'labels'/side).mkdir(parents=True)
        for r in rs:
            dest=folder/'images'/side/(r['id']+Path(r['path']).suffix.lower())
            shutil.copy2(r['path'],dest)
            text=''.join(f"{CLASS_MAP[c.lower()]} {x} {y} {w} {h}\n" for c,x,y,w,h in r['boxes'])
            (folder/'labels'/side/(r['id']+'.txt')).write_text(text)
    data=folder/'data.yaml'
    data.write_text(yaml.safe_dump(dict(path=str(folder), train='images/train',val='images/val',test='images/test',names=OUR_NAMES)))
    return data

def image_scores(model, records, folder, subsets=('all','cobweb','non-cobweb')):
    folder.mkdir(parents=True,exist_ok=True)
    predictions=[]
    for r in records:
        res=model.predict(r['path'],conf=.001,imgsz=TRAIN_SETTINGS['imgsz'],iou=.7,max_det=300,verbose=False)[0]
        boxes=res.boxes
        predictions.append(dict(id=r['id'],original_filename=r['original_filename'],path=r['path'],
            positive=r['positive'] if r['benchmark'] else bool(r['boxes']),has_cobweb=r['has_cobweb'],
            group=r['group'],benchmark_category=r.get('benchmark_category'),
            label_basis=r.get('label_basis','boxes_or_verified_clean'),confidences=boxes.conf.cpu().tolist(),
            classes=boxes.cls.cpu().tolist(),xyxy=boxes.xyxy.cpu().tolist()))
    save_json(folder/'predictions.json',predictions)
    rows=[]
    for subset in subsets:
        ps=[p for p in predictions if subset=='all' or (p['has_cobweb'] if subset=='cobweb' else not p['has_cobweb'])]
        for conf in (.25,.50):
            pos=[p for p in ps if p['positive']]; clean=[p for p in ps if not p['positive']]
            tp=sum(any(c>=conf for c in p['confidences']) for p in pos)
            fp=sum(any(c>=conf for c in p['confidences']) for p in clean)
            rows.append(dict(subset=subset,confidence=conf,n_images=len(ps),n_positive=len(pos),n_clean=len(clean),
                detected_positive=tp,false_alarms=fp,sensitivity=tp/len(pos) if pos else None,
                clean_false_alarm_rate=fp/len(clean) if clean else None))
    pd.DataFrame(rows).to_csv(folder/'image_metrics.csv',index=False)
    # Qualitative failures at both requested thresholds.
    for conf in (.25,.50):
        failures=[p for p in predictions if any(c>=conf for c in p['confidences'])!=p['positive']]
        save_json(folder/f'failures_{conf:.2f}.json',failures)
        for p in failures[:12]:
            with Image.open(p['path']) as im: im=im.convert('RGB')
            d=ImageDraw.Draw(im)
            for xy,c in zip(p['xyxy'],p['confidences']):
                if c>=conf: d.rectangle(xy,outline='red',width=3)
            im.thumbnail((800,800)); im.save(folder/f"failure_{conf:.2f}_{p['id']}.jpg")
    return rows

def detection_scores(model, records, folder):
    rows=[]
    for subset in ('all','cobweb','non-cobweb'):
        rs=[r for r in records if subset=='all' or (r['has_cobweb'] if subset=='cobweb' else not r['has_cobweb'])]
        nboxes=sum(len(r['boxes']) for r in rs)
        row=dict(subset=subset,n_images=len(rs),n_boxes=nboxes,n_clean=sum(r['clean'] for r in rs),
            precision=None,recall=None,mAP50=None)
        if nboxes:
            data=build_dataset(folder/subset,dict(train=[],val=[],test=rs))
            m=model.val(data=str(data),split='test',imgsz=TRAIN_SETTINGS['imgsz'],batch=16,device=0,
                conf=.001,iou=.7,max_det=300,plots=True,project=str(folder),name='plots_'+subset,exist_ok=False)
            row.update(precision=float(m.box.mp),recall=float(m.box.mr),mAP50=float(m.box.map50),
                per_class_AP50={OUR_NAMES[int(c)]:float(a) for c,a in zip(m.box.ap_class_index,m.box.ap50)})
        rows.append(row)
    return rows

def run_pair():
    if not VALIDATED: raise ValueError('Run successful label/split validation first')
    validate_splits(unique)
    if json.loads(contract_file.read_text())!=contract:
        raise ValueError('Split contract changed after validation; rerun preparation')
    if hashlib.sha256(json.dumps(raw+bench,sort_keys=True).encode()).hexdigest()!=VALIDATED_DIGEST:
        raise ValueError('Records changed after validation; rerun audit/splits')
    for r in raw+bench:
        if sha(r['path'])!=r['image_sha256'] or (r['label_path'] and sha(r['label_path'])!=r['label_sha256']):
            raise ValueError('Input image/label changed after audit; rerun preparation')
    if not GROUPING_REVIEWED: raise ValueError('Review dish/sequence IDs, duplicate grids and labels; set GROUPING_REVIEWED=True')
    if any(RF_VERSIONS[s]!=resolved[s] for s in SETS): raise ValueError('Pin RF_VERSIONS to the audited versions first')
    if KAGGLE_VERSION=='FILL_ME': raise ValueError('Record Kaggle notebook version')
    from ultralytics import YOLO
    import torch
    # Load once and save the exact initial checkpoint for both experiments.
    initial=OUT/'initial.pt'
    if initial.exists(): raise ValueError('Comparison already started; choose a new ID or inspect saved outputs')
    YOLO(MODEL).save(str(initial))
    settings=dict(train=TRAIN_SETTINGS,source_versions=resolved,source_projects=SETS,model=MODEL,initial_sha256=sha(initial),
        split_contract_sha256=sha(contract_file),kaggle_version=KAGGLE_VERSION,
        python=sys.version,platform=platform.platform(),grouping_reviewed=GROUPING_REVIEWED,
        date=pd.Timestamp.now(tz="UTC").isoformat(),torch=torch.__version__,cuda=torch.version.cuda,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    save_json(OUT/'settings.json',settings)
    (OUT/'requirements.lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    held={side:[r for r in unique if r['split']==side] for side in ('train','val','test')}
    log=[]
    for tag in ('with-cobweb','without-cobweb'):
        run=OUT/tag; run.mkdir()
        train=[r for r in held['train'] if tag=='with-cobweb' or not r['has_cobweb']]
        parts=dict(train=train,val=held['val'],test=held['test'])
        # Per-run manifests preserve original names, including excluded photos.
        pd.DataFrame([dict(x,included=(x['split']!='train' or tag=='with-cobweb' or not x['has_cobweb'])) for x in manifest]).to_csv(run/'split_manifest.csv',index=False)
        shutil.copy2(OUT/'benchmark_manifest.csv',run/'benchmark_manifest.csv')
        (run/'test_files.txt').write_text('\n'.join(r['path'] for r in held['test']))
        save_json(run/'settings.json',settings|{'cobweb':tag,'n_train_photos':len(train)})
        (run/'notes.md').write_text(f"Mould check; {tag}. Kaggle version {KAGGLE_VERSION}; {settings['date']}.\nSame fixed validation and test groups; patience=15.\nBaibua one-owner data; clean d005 source confounding; folder benchmark is image-level only.\nSmall cobweb subsets limit conclusions; syldd excluded; myceliummonitor excluded.\n")
        try:
            data=build_dataset(WORK/'datasets'/COMPARISON_ID/tag,parts)
            YOLO(str(initial)).train(data=str(data),project=str(run),name='training',exist_ok=False,**TRAIN_SETTINGS)
            best=run/'training'/'weights'/'best.pt'
            shutil.copy2(best,run/'best.pt')
            model=YOLO(str(run/'best.pt'))
            scores={'detection':detection_scores(model,held['test'],run/'detection'),
                'image_level':image_scores(model,held['test'],run/'test_image_level')}
            scores['benchmark']={}
            train_groups={r['group'] for r in held['train']+held['val']}
            for kind in ('unmatched','overlap-heldout','overlap-training-or-tuning'):
                rs=[r for r in bench if ('overlap-training-or-tuning' if r['group'] in train_groups else 'overlap-heldout' if r['benchmark_overlap'] and any(x['group']==r['group'] for x in unique) else 'unmatched')==kind]
                # Folder labels identify cobweb photos, but supply no reference boxes.
                scores['benchmark'][kind]=image_scores(model,rs,run/'benchmark'/kind)
            save_json(run/'scores.json',scores)
            a=scores['detection'][0]; rates={r['confidence']:r for r in scores['image_level'] if r['subset']=='all'}
            log.append(dict(run_id=COMPARISON_ID+'_'+tag,model='Mould check',date=pd.Timestamp.now(tz='UTC').isoformat(),
                notebook='2.6.kaggle_mould_detection.ipynb',kaggle_version=KAGGLE_VERSION,train_sets='+'.join(SETS)+'+d005',
                test_set='fixed grouped test',cobweb=tag,epochs=len(pd.read_csv(run/'training'/'results.csv')),
                imgsz=TRAIN_SETTINGS['imgsz'],seed=SEED,precision=a['precision'],recall=a['recall'],mAP50=a['mAP50'],
                **{'false_alarm_0.25':rates[.25]['clean_false_alarm_rate'],'false_alarm_0.50':rates[.5]['clean_false_alarm_rate']},
                n_train_photos=len(train),n_test_photos=len(held['test']),notes='See bundled notes and audit'))
            pd.DataFrame(log).to_csv(OUT/'run_log.csv',index=False)
        except Exception as e:
            (run/'failure.txt').write_text(repr(e)); raise
        finally:
            # Save after each run, including partial outputs on failure; never delete run evidence.
            shutil.make_archive(str(OUT)+'_'+tag,'zip',OUT)
    display(pd.DataFrame(log))

if RUN_TRAINING:
    run_pair()
else:
    print('Training disabled. Audit only; review output and pin source versions before enabling.')


## Kaggle run and download
1. Upload/publish the updated `mould-test-photos` Kaggle dataset version containing `clean/`, `cobweb/`, and `mould/`; select that input version in this notebook. Use a fresh `COMPARISON_ID` if an audit or run already exists: moved photos and folder labels change the frozen contract. Audit with training off. Verify both input paths and benchmark folder counts, original class/cobweb counts, annotation grids, duplicate groups, clean dish IDs, benchmark overlap and all three split counts. Correct dish/sequence overrides before freezing the split. Fix rejected labels at the source; do not bypass validation. Retain the audit ZIP.
2. Pin printed Roboflow versions. Set `GROUPING_REVIEWED=True`, fill `KAGGLE_VERSION`, then enable `RUN_TRAINING`. A first optional two-epoch smoke run needs its **own comparison ID** and `epochs=2`; verify bundles/metrics, then use a fresh ID with 50 epochs and patience 15. Smoke results are not final evidence. No automatic resume or overwrite of started comparisons.
3. Click **Save Version → Save & Run All (Commit)** for a persistent run. Download the completed version's Output ZIPs from `/kaggle/working/cv_runs/mould/`. Each ZIP contains both completed runs so far, weights, epoch histories/plots, test metrics/predictions/failures, settings/package lock, provenance, split manifests and notes. A failure ZIP preserves progress. Some attached/raw images may also appear in Output; no automatic deletion is performed.
4. Extract locally to `kaggle/cv_runs/mould/<comparison_id>/`; retain `run_log.csv`. Screenshot the final comparison table. To reuse a model, publish the notebook output as a private Kaggle dataset, attach it, and load that run's `best.pt`. Keep the original notebook version and source versions with the bundle.

Kaggle must verify downloads/licences, actual Baibua label/dish identities, full overlap audit, CUDA/checkpoint compatibility, short smoke run, early stopping, metric files and output persistence. Re-running on different hardware/software may change floating-point results; fixed manifests, initial weights, package lock and settings preserve the recipe. Do not tune on test/benchmark or syldd. Myceliummonitor needs a separate notebook/run protocol and independent provenance/overlap audit.
